In [1]:
import torch, torch.nn as nn
print(torch.__version__)

2.11.0+cu130


In [8]:
#Cell E1: one neuron (this is the whole idea)
x = torch.tensor([2.0, -1.0, 0.5])
w = torch.tensor([0.4, 0.3, -0.2])
b = torch.tensor(0.1)
z = (w * x).sum() + b
print(z)
print(torch.relu(z))
print(torch.sigmoid(z))

tensor(0.5000)
tensor(0.5000)
tensor(0.6225)


- A neuron takes a set of input features, multiplies each by a corresponding weight, adds a bias term to compute a single weighted sum, and then passes that result through an activation function to generate its final output signal.

In [9]:
#Cell E2: three activation functions
t = torch.tensor([-2., -1., 0., 1., 2.])
print(torch.relu(t))
print(torch.sigmoid(t).round(decimals=4))

logits = torch.tensor([2.0, 1.0, 0.1])
sm = torch.softmax(logits, dim=0)
print(sm.round(decimals=4), sm.sum())

tensor([0., 0., 0., 1., 2.])
tensor([0.1192, 0.2689, 0.5000, 0.7311, 0.8808])
tensor([0.6590, 0.2424, 0.0986]) tensor(1.0000)


- ReLU does capped the first three numbers at t 0 because the numbers are negative and one of it is 0.

- Sigmoid's output range squashes into a strict probability 0 to 1. It is used exclusively for Binary Classification tasks (like predicting whether a customer will churn or not, or if an individual earns above or below $50K), where the output represents the probability of the sample belonging to the positive class (1).

- It turns the logits number into probability that add to 1.  It is used exclusively for Multi-Class Classification tasks (such as classifying an image as a cat, dog, or bird), where it transforms a vector of raw model scores into a valid probability distribution across three or more mutually exclusive categories.

In [ ]:
#Cell E3: a layer is many neurons side by side
W1 = torch.tensor([[0.4, 0.3, -0.2],
                   [-0.1, 0.2, 0.5]])
b1 = torch.tensor([0.1, -0.3])
layer = nn.Linear(3, 2)
with torch.no_grad():
  layer.weight.copy_(W1)
  layer.bias.copy_(b1)
print(layer(x))
print(x @ W1.T + b1)
# print(x @ W1.T + b)    # my experiment: b is the single bias 0.1 from E1 -> broadcasting gave [0.5, -0.05], no error
print(layer.weight.shape, layer.bias.shape, sum(p.numel() for p in layer.parameters()))

tensor([ 0.5000, -0.4500], grad_fn=<ViewBackward0>)
tensor([ 0.5000, -0.0500])
torch.Size([2, 3]) torch.Size([2]) 8


- PyTorch stores the weight as (out_features, in_features) = (2, 3): each row is one neuron’s 3 weights. The layer then computes x @ W.T, which is why the code transposes it.

In [16]:
#Cell E4: the forward pass of a tiny network: 3 inputs -> 2 hidden neurons (ReLU) -> 1 output (sigmoid)
W2 = torch.tensor([[0.6, -0.4]])
b2 = torch.tensor([0.1])
h = torch.relu(layer(x))
z2 = h @ W2.T + b2
p = torch.sigmoid(z2)
print(h)
print(z2)
print(p)

tensor([0.5000, 0.0000], grad_fn=<ReluBackward0>)
tensor([0.4000], grad_fn=<AddBackward0>)
tensor([0.5987], grad_fn=<SigmoidBackward0>)


In [20]:
#Cell E5: the loss function (binary classification)
y = torch.tensor([1.0])
manual = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))
print(manual)
print(nn.BCELoss()(p, y))
print(nn.BCEWithLogitsLoss()(z2, y))

for q in (0.99, 0.5, 0.01):
  print(q, round(-torch.log(torch.tensor(q)).item(), 4))

tensor([0.5130], grad_fn=<NegBackward0>)
tensor(0.5130, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.5130, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
0.99 0.0101
0.5 0.6931
0.01 4.6052


- The last loop show about being confidently wrong (true y = 1) is that showing how neural network punishes when the prediction is wrong because Binary Cross-Entropy loss heavily penalizes the model by spiking up exponentially to 4.6052.

In [22]:
#Cell E6: backpropagation (autograd does the chain rule for the whole network)
W1p = W1.clone().requires_grad_()
b1p = b1.clone().requires_grad_()
W2p = W2.clone().requires_grad_()
b2p = b2.clone().requires_grad_()
h = torch.relu(x @ W1p.T + b1p)
z2 = h @ W2p.T + b2p
loss = nn.BCEWithLogitsLoss()(z2, y)
loss.backward()
print(loss.item())
print(b2p.grad)
print(W2p.grad)
print(W1p.grad)
print(b1p.grad)

0.5130152702331543
tensor([-0.4013])
tensor([[-0.2007,  0.0000]])
tensor([[-0.4816,  0.2408, -0.1204],
        [ 0.0000, -0.0000,  0.0000]])
tensor([-0.2408,  0.0000])


- (a) b2p.grad = p - y = 0.5987 - 1 = -0.4013

- (b) W2p.grad[0,0] = (p - y) x h1 = -0.4013 x 0.5 = -0.2007

- (c) W1p.grad[0] = (p - y) x W2[0,0] x x = -0.4013 x 0.6 x [2, -1, 0.5]

- The entire second row of W1p.grad is zero because hidden neuron 2 computed a negative pre-activation value of z = -0.45, which caused the ReLU activation function to completely clip its output to exactly 0